### 추론 모델 (Reasoning Models)
추론(reasoning) 모델은 강화학습(Reinforcement Learning)을 통해 논리적 사고와 문제 해결 능력을 학습한 대형 언어 모델(LLM)입니다.

이 모델들은 답변을 바로 생성하지 않고, 응답하기 전에 내부적으로 긴 사고 과정(chain of thought) 을 거친 후 최종 답변을 만듭니다.
그 결과, 복잡한 문제 해결, 코딩, 과학적 추론, 다단계 계획이 필요한 에이전트 기반 워크플로우(agentic workflows) 에서 뛰어난 성능을 보입니다.

이 모델들은 또한 OpenAI의 경량 코딩 에이전트인 Codex CLI 환경에서 가장 우수한 모델로 사용됩니다. 

| 모델                | 특징                                                       |
| ----------------- | -------------------------------------------------------- |
| **gpt-5-nano**    | 매우 빠르고 저비용, 경량 모델                                        |
| **gpt-5-mini**    | 속도와 품질의 균형형 모델                                           |
| **gpt-5 (기본 모델)** | 가장 크고 정교하며, 복잡한 작업과 다양한 도메인에서 최고의 품질 제공 (속도는 느리고 비용은 높음) |


In [1]:
from dotenv import load_dotenv
load_dotenv() 

True

In [2]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-5-nano"

In [3]:
prompt = """
문자열로 표현된 행렬을 '[1,2],[3,4],[5,6]' 형식으로 받아서 같은 형식으로 전치행렬을 출력하는 파이썬 스크립트를 작성하세요.
"""

response = client.responses.create(
    model="gpt-5",
    reasoning={"effort": "medium"},
    input=[
        {
            "role": "user", 
            "content": prompt
        }
    ]
)

print(response.output_text)

아래와 같은 간단한 파이썬 스크립트로 구현할 수 있습니다.  
문자열 `"[1,2],[3,4],[5,6]"` 을 입력으로 받아 전치행렬을 동일한 포맷으로 출력합니다.

```python
#!/usr/bin/env python3
import sys

def transpose_str_matrix(s: str) -> str:
    # 공백 제거
    s = s.replace(" ", "")
    # “[1,2],[3,4],[5,6]” -> [[1,2],[3,4],[5,6]] 형태로 eval
    matrix = eval(f"[{s}]")
    # 전치
    transposed = zip(*matrix)
    # "[1,3,5],[2,4,6]" 형태의 문자열로 포맷팅
    return ",".join(
        "[" + ",".join(map(str, row)) + "]"
        for row in transposed
    )

if __name__ == "__main__":
    # 표준 입력에서 한 줄 읽기
    line = sys.stdin.readline().strip()
    print(transpose_str_matrix(line))
```

실행 예시:

```
$ echo "[1,2],[3,4],[5,6]" | python3 transpose.py
[1,3,5],[2,4,6]
```

—  
주의: 위 코드는 매우 간단화를 위해 `eval` 을 사용하고 있으므로, 외부 입력을 임의의 코드 실행으로부터 보호할 필요가 있는 환경에서는 별도의 파서(예: `re` 모듈로 숫자만 추출) 등을 사용하는 편이 안전합니다.


### 추론 모델의 비용 관리  

비용을 관리하려면 `max_output_tokens` 매개변수를 사용하여 모델이 생성하는 총 토큰 수(추론 토큰과 응답 토큰 포함)를 제한할 수 있습니다.  

In [4]:
prompt = """
문자열로 표현된 행렬을 '[1,2],[3,4],[5,6]' 형식으로 받아서 같은 형식으로 전치행렬을 출력하는 파이썬 스크립트를 작성하세요.
"""

response = client.responses.create(
    model="gpt-5-nano",
    reasoning={"effort": "medium"},
    input=[
        {
            "role": "user", 
            "content": prompt
        }
    ],
    max_output_tokens=300,
)

if response.status == "incomplete" and response.incomplete_details.reason == "max_output_tokens":
    print("Ran out of tokens")
    if response.output_text:
        print("Partial output:", response.output_text)
    else: 
        print("추론 과정에서 토큰이 소진되었습니다.")

Ran out of tokens
추론 과정에서 토큰이 소진되었습니다.


- 특정 출력을 생성하기 위한 명확한 지침이 있을 때 가장 좋은 성과를 낼 수 있습니다.

In [5]:
from openai import OpenAI

client = OpenAI()

prompt = """
Instructions:
- 아래 주어진 React 컴포넌트를 수정하여, 'nonfiction' 카테고리의 책 제목은 빨간색 글씨로 표시되도록 하시오.
- 답변에는 코드만 포함하시오.
- 마크다운 코드 블록 같은 추가 포맷팅은 포함하지 마시오.
- 코드 들여쓰기는 4개의 스페이스를 사용하고, 한 줄이 80자를 초과하지 않도록 하시오.

const books = [
  { title: 'Dune', category: 'fiction', id: 1 },
  { title: 'Frankenstein', category: 'fiction', id: 2 },
  { title: 'Moneyball', category: 'nonfiction', id: 3 },
];

export default function BookList() {
  const listItems = books.map(book =>
    <li>
      {book.title}
    </li>
  );

  return (
    <ul>{listItems}</ul>
  );
}
"""

response = client.responses.create(
    model="gpt-5-nano",
    input=[
        {
            "role": "user",
            "content": prompt,
        }
    ]
)

print(response.output_text)

const books = [
    {
        title: 'Dune',
        category: 'fiction',
        id: 1
    },
    {
        title: 'Frankenstein',
        category: 'fiction',
        id: 2
    },
    {
        title: 'Moneyball',
        category: 'nonfiction',
        id: 3
    }
];

export default function BookList() {
    const listItems = books.map(book =>
        <li
            key={book.id}
            style={{
                color: book.category === 'nonfiction' ? 'red' : undefined
            }}
        >
            {book.title}
        </li>
    );

    return (
        <ul>{listItems}</ul>
    );
}
